# AI Voice Feedback AgentAgentic flow for collecting customer feedback via a short (45s) voice call.**Flow:**1. Business profile input (business, target user, tone, feedback objective) -- all free text2. LLM proposes feedback approaches and you **negotiate in plain English** until one is right3. Language -- say it however you like ("Hindi", "brazilian portuguese"); the LLM resolves it4. STT/TTS engines (pluggable; console mock until real keys are added)5. Real-time dialogue manager -- handles silence, off-topic questions, "who are you", within a hard 45s budget6. Run the simulated call7. Structured feedback extraction from the transcriptNothing is a fixed menu. There are no numbered options to pick from and no hardcoded list of languages or feedback types -- you describe what you want and the model decides how to satisfy it.**Latency design (a voice call cannot wait on a slow model):**- `gemini-3.5-flash-lite` with `thinking_level="low"` for every in-call turn (~0.9s); the bigger `gemini-3.5-flash` only for the offline setup/summary steps, where a second costs nothing.- **Streamed turns**: the reply is spoken the moment the `speech` field closes, without waiting for the rest of the JSON.- A hard `response_schema`, so the model cannot return prose we then have to repair.- Small `max_output_tokens` on turns, a warm-up request to pay TLS/handshake cost before the call starts, and per-turn timings printed so regressions are visible.**Credentials** -- resolved from Colab Secrets, then the environment/`.env`, then a prompt, so this notebook runs unchanged in both places:- **Colab**: click the key icon in the left sidebar, add a secret named exactly `GEMINI_API_KEY`, and toggle notebook access on.- **Local**: put it in a `.env` file next to this notebook.Use a real API key (starts with `AIza`) from [aistudio.google.com/apikey](https://aistudio.google.com/apikey). An AI Studio *ephemeral* token (starts with `AQ.`) is short-lived and bound to the session that issued it -- it will 401 from Colab even while it still works on your own machine.Implement the two marked methods in Section 4 to go from mock to real audio -- nothing else in the flow changes.

In [ ]:
# Note: quote the version specs -- unquoted `>=` is a shell redirect and silently
# creates files named "0.8.0", "1.26.0", ... instead of installing anything.
# On Colab, -U matters: the preinstalled google-genai is usually too old for
# thinking_level, which is what keeps the in-call turns fast.
%pip install -q -U "google-genai>=1.0.0" "python-dotenv>=1.0.0"

## 0. Setup

In [ ]:
import json, os, re, sys, time
from dataclasses import dataclass, field
from getpass import getpass
from typing import Dict, List, Optional

from dotenv import find_dotenv, load_dotenv
from google import genai
from google.genai import types
from google.genai import errors as genai_errors

IN_COLAB = "google.colab" in sys.modules

# override=True matters: load_dotenv() defaults to override=False, so once a stale
# key is in os.environ (e.g. you edited .env after the kernel started) it silently
# wins over the file and you get a confusing 401. Re-running this cell now always
# picks up the current .env without restarting the kernel.
DOTENV_PATH = find_dotenv(usecwd=True)
if DOTENV_PATH:
    load_dotenv(DOTENV_PATH, override=True)


def get_secret(name: str, required: bool = False) -> str:
    """Resolve a credential across every place it might live, so the same notebook
    runs locally and on Colab unchanged. Never hardcode keys in the notebook itself --
    they leak the moment the .ipynb is shared.

    Order: Colab Secrets (key icon in the left sidebar) -> environment/.env -> prompt.
    """
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = (userdata.get(name) or "").strip()
            if val:
                return val
        except Exception:
            pass  # secret not set, or notebook not granted access to it
    val = (os.getenv(name) or "").strip()
    if val:
        return val
    if required:
        if IN_COLAB:
            print(f"{name} not found in Colab Secrets. Add it via the key icon in the left "
                  f"sidebar (name it exactly {name} and enable notebook access), or paste it below.")
        val = getpass(f"{name}: ").strip()
        if val:
            return val
        raise RuntimeError(
            f"{name} missing. Local: put it in .env next to the notebook. "
            f"Colab: add it under Secrets (key icon) as {name}.")
    return ""


GEMINI_API_KEY = get_secret("GEMINI_API_KEY", required=True)
STT_API_KEY = get_secret("STT_API_KEY")
TTS_API_KEY = get_secret("TTS_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

# Two model tiers. In-call turns are latency-critical; the offline setup/summary
# steps are not, so they can afford the bigger model.
TURN_MODEL = "gemini-3.5-flash-lite"   # ~0.9s per turn with thinking low
# Also lite by default: the free tier caps gemini-3.5-flash at 20 requests/DAY, and one
# pass through this notebook spends several here. Switch to "gemini-3.5-flash" for
# stronger negotiation/summaries once billing is enabled. See section 0b.
PLANNING_MODEL = "gemini-3.5-flash-lite"

TEST_MODE = True   # True = no input() anywhere: canned answers + scripted customer replies
VERBOSE_LATENCY = True  # print per-turn model latency

# Thinking costs ~1s+ per turn, which is dead air on a phone call. "low" is the
# floor this model family accepts and is plenty for short scripted dialogue.
FAST_THINKING = types.ThinkingConfig(thinking_level="low")


def json_config(schema: types.Schema, max_tokens: int, system: Optional[str] = None,
                temperature: float = 0.3) -> types.GenerateContentConfig:
    """Config that forces schema-valid JSON, so no output parsing/repair is ever needed."""
    return types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        max_output_tokens=max_tokens,
        response_mime_type="application/json",
        response_schema=schema,
        thinking_config=FAST_THINKING,
    )


def with_retry(fn, attempts: int = 3):
    """Retry past rate limits. The free tier is stingy (as low as 20 requests/day on the
    bigger models), and a 429 mid-negotiation would otherwise lose the whole conversation.
    Honours the server's retryDelay when it gives one."""
    for i in range(attempts):
        try:
            return fn()
        except genai_errors.ClientError as e:
            msg = str(e)
            if getattr(e, "code", None) != 429:
                raise
            if i == attempts - 1:
                raise RuntimeError(
                    "Gemini rate limit hit and retries exhausted.\n"
                    "  The free tier allows very few requests/day on the larger models.\n"
                    "  Options: wait for the quota to reset, point PLANNING_MODEL at a "
                    "'-lite' model (they have far higher free limits), or enable billing.\n"
                    "  Check usage at https://ai.dev/rate-limit\n"
                    f"  Original error: {msg[:300]}") from None
            m = re.search(r"retryDelay['\"]?:\s*['\"]?(\d+)", msg)
            wait = min(float(m.group(1)) + 1 if m else 5 * (i + 1), 65)
            print(f"  rate limited, retrying in {wait:.0f}s...")
            time.sleep(wait)


def call_json(model: str, prompt: str, schema: types.Schema, max_tokens: int = 2048,
              system: Optional[str] = None) -> dict:
    """Single-shot JSON call with a readable error if the model output was truncated."""
    resp = with_retry(lambda: client.models.generate_content(
        model=model, contents=prompt, config=json_config(schema, max_tokens, system)))
    text = (resp.text or "").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        finish = resp.candidates[0].finish_reason if resp.candidates else "unknown"
        raise ValueError(
            f"Model returned invalid/truncated JSON (finish_reason={finish}). "
            f"Raise max_tokens for this call. Raw output: {text!r}") from e


def warm_up() -> float:
    """Preflight: pay TLS handshake + connection setup now, not on the customer's first
    turn, and fail here with an actionable message rather than mid-call."""
    t0 = time.perf_counter()
    try:
        client.models.generate_content(
            model=TURN_MODEL, contents="ok",
            config=types.GenerateContentConfig(max_output_tokens=1, thinking_config=FAST_THINKING))
    except genai_errors.ClientError as e:
        if getattr(e, "code", None) in (401, 403):
            if GEMINI_API_KEY.startswith("AQ."):
                hint = ("This is an AI Studio EPHEMERAL token ('AQ.' prefix), not an API key. It is "
                        "short-lived and tied to the session that minted it, so it commonly works on "
                        "your own machine and still fails 401 from Colab. Create a real API key "
                        "(starts with 'AIza') at https://aistudio.google.com/apikey")
            elif not GEMINI_API_KEY.startswith("AIza"):
                hint = ("A Gemini API key normally starts with 'AIza'. Get one at "
                        "https://aistudio.google.com/apikey")
            else:
                hint = ("Key looks well-formed, so it is likely revoked, or the Generative Language "
                        "API is not enabled on its project. Re-issue it at "
                        "https://aistudio.google.com/apikey")
            where = ("Colab Secrets (key icon in the left sidebar)" if IN_COLAB
                     else f".env at {DOTENV_PATH or '<none found>'}")
            raise RuntimeError(
                f"Gemini rejected the credentials ({e.code}).\n"
                f"  running on: {'Google Colab' if IN_COLAB else 'local kernel'}\n"
                f"  key source: {where}\n"
                f"  key loaded: {GEMINI_API_KEY[:6]}...{GEMINI_API_KEY[-4:]} (len {len(GEMINI_API_KEY)})\n"
                f"  {hint}\n"
                f"  After updating the key, re-run THIS cell -- it re-reads the secret/.env.") from None
        raise
    return time.perf_counter() - t0


print("Gemini configured | turn model:", TURN_MODEL, "| planning model:", PLANNING_MODEL)
print(f"Environment: {'Google Colab' if IN_COLAB else 'local kernel'}")
print(f"Key loaded: {GEMINI_API_KEY[:6]}...{GEMINI_API_KEY[-4:]} (len {len(GEMINI_API_KEY)})")
if GEMINI_API_KEY.startswith("AQ."):
    print("  WARNING: 'AQ.' is an ephemeral AI Studio token, not an API key. It expires quickly")
    print("           and is bound to its origin session -- expect 401s, especially on Colab.")
    print("           Create a durable key (starts with 'AIza') at https://aistudio.google.com/apikey")
print("STT key present:", bool(STT_API_KEY), "| TTS key present:", bool(TTS_API_KEY))
print(f"Warm-up round trip: {warm_up():.2f}s")

### 0b. Which models can I use? (optional)Model names change and old ones get retired, so don't guess -- this asks your key what it can actually reach.To switch models, edit the two constants in the Setup cell above:- **`TURN_MODEL`** runs every in-call turn, so it dominates how snappy the call feels. Keep it a `-lite` / small model.- **`PLANNING_MODEL`** runs the setup negotiation and the final summary. These are off the clock, so favour capability here.If a name is retired, the API says so and names its replacement in the error, e.g. *"models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash"*.**Watch the free-tier quota.** Non-lite models have a very low free daily cap (`gemini-3.5-flash` allows **20 requests/day**), and one full pass through this notebook spends several on `PLANNING_MODEL` — the negotiation alone costs one per round. If you hit `429 RESOURCE_EXHAUSTED`, either wait for the reset, point `PLANNING_MODEL` at a `-lite` model (much higher free limits, slightly weaker negotiation), or enable billing. Live usage: [ai.dev/rate-limit](https://ai.dev/rate-limit).

In [ ]:
# Ask the API what this key can reach, rather than trusting a hardcoded list.
available = [m.name.removeprefix("models/") for m in client.models.list()
             if "generateContent" in (m.supported_actions or [])]

print(f"{len(available)} models available. Text models with 'flash' or 'pro' in the name:\n")
for name in sorted(n for n in available if ("flash" in n or "pro" in n)
                   and not any(x in n for x in ("image", "tts", "embedding", "vision"))):
    marks = []
    if name == TURN_MODEL:
        marks.append("<- TURN_MODEL")
    if name == PLANNING_MODEL:
        marks.append("<- PLANNING_MODEL")
    print(f"  {name:42} {' '.join(marks)}")

for label, chosen in (("TURN_MODEL", TURN_MODEL), ("PLANNING_MODEL", PLANNING_MODEL)):
    if chosen not in available:
        print(f"\nWARNING: {label}={chosen!r} is not in this key's model list; calls will 404.")


def benchmark(model_name: str, runs: int = 3) -> float:
    """Time a realistic short JSON turn. Use this before promoting a model to TURN_MODEL --
    anything much over ~1s is audible dead air on a phone call."""
    cfg = types.GenerateContentConfig(
        max_output_tokens=TURN_MAX_TOKENS if "TURN_MAX_TOKENS" in globals() else 160,
        temperature=0.4, thinking_config=FAST_THINKING,
        response_mime_type="application/json")
    best = float("inf")
    for _ in range(runs):
        t0 = time.perf_counter()
        client.models.generate_content(
            model=model_name, config=cfg,
            contents='Return JSON {"speech": <one short spoken sentence asking a cafe '
                     'customer to rate their visit 1 to 5>, "should_end": false}')
        best = min(best, time.perf_counter() - t0)
    return best

# Uncomment to compare candidates before switching:
# for m in ["gemini-3.5-flash-lite", "gemini-3.5-flash"]:
#     print(f"{m:28} best of 3: {benchmark(m):.2f}s")

## 1. Business Profile InputEvery field is free text typed by the business user. `TEST_MODE = True` substitutes the bracketed defaults so the whole notebook runs top to bottom without stopping for input.

In [ ]:
def ask(prompt: str, default: str = "") -> str:
    """Free-text prompt for the business user. In TEST_MODE, silently take the default."""
    if TEST_MODE:
        return default
    shown = f"{prompt} [{default}]: " if default else f"{prompt}: "
    while True:
        val = input(shown).strip()
        if val or default:
            return val or default
        print("  (this one is required)")


def ask_int(prompt: str, default: int, lo: int, hi: int) -> int:
    """Numeric prompt that re-asks instead of crashing on junk input."""
    if TEST_MODE:
        return default
    while True:
        raw = input(f"{prompt} [{default}]: ").strip()
        if not raw:
            return default
        if raw.isdigit() and lo <= int(raw) <= hi:
            return int(raw)
        print(f"  (enter a whole number from {lo} to {hi})")


business_profile = {
    "business_name": ask("Business name", "Bella Vista Cafe"),
    "business_description": ask("What does the business do?", "A neighborhood cafe serving coffee, breakfast, and lunch."),
    "target_user": ask("Who is being called?", "Walk-in customers who just finished dining or picking up an order."),
    "tone": ask("Desired call tone", "warm, friendly, concise"),
    "feedback_objective": ask("What do you want to learn from this feedback?", "Understand satisfaction with food quality and service speed, and catch any recurring complaints."),
}

for k, v in business_profile.items():
    print(f"{k:22} {v}")

## 2. Agree on a Feedback Approach (conversational)The model proposes a few feedback approaches. You are **not** picking a list index -- you reply in plain English:- `ok` / `yes` / Enter -- accept what it is proposing- *"use the second one but make it about ambience"* -- refine it- *"none of these, I want to know if they'd come back"* -- ask for something different entirelyIt keeps going until it locks in a single approach, and it decides what that approach is -- the scale, the wording, and how to score it are all its call, not a hardcoded enum.**The channel constrains what it may propose.** This is a phone call and nothing else: the customer has a phone at their ear and no screen. So every approach must be answerable by *speaking* or by *pressing a digit* (DTMF) -- never a thumbs up/down, a button, a star to tap, a link, or an SMS. Each proposal declares an `input_mode` (`speech` / `keypad` / `either`) and, when keys are involved, the mapping it will read aloud. Ask it for something the channel can't do and it will say so and offer the closest workable equivalent.Keypad answers are worth having: they survive noisy lines, heavy accents, and STT failures, which is exactly when a spoken rating gets lost.

In [ ]:
# Note there is no fixed list of dimension types. The model invents whatever scale
# fits what the business user asked for, and tells us how to score it.
#
# input_mode is the constraint that keeps proposals physically possible: this is a
# phone call, so the customer can only speak or press keys. There is no screen, so
# no thumbs up/down, no buttons, no stars to tap, no links.
APPROACH_FIELDS = {
    "name": types.Schema(type="STRING", description="Short label for this approach"),
    "description": types.Schema(type="STRING", description="One sentence, under 20 words"),
    "dimension_type": types.Schema(type="STRING", description="Machine-ish slug for the scale, e.g. rating_1_5, nps_0_10, yes_no, open_ended"),
    "input_mode": types.Schema(type="STRING", enum=["speech", "keypad", "either"],
                               description="How the customer answers: speaking, pressing digits, or either"),
    "sample_question": types.Schema(type="STRING", description="The question as spoken aloud, including the keypress instruction when input_mode is keypad"),
    "keypad_map": types.Schema(type="STRING", nullable=True,
                               description="What each digit means, e.g. '1-5 = rating, 1 worst 5 best'. Null when input_mode is speech."),
    "score_guidance": types.Schema(type="STRING", description="One sentence telling a later summarizer exactly what 'score' means here, or that it stays null"),
}
APPROACH_SCHEMA = types.Schema(type="OBJECT", properties=APPROACH_FIELDS,
                               required=["name", "description", "dimension_type",
                                         "input_mode", "sample_question", "score_guidance"])

NEGOTIATION_SCHEMA = types.Schema(
    type="OBJECT",
    properties={
        "reply": types.Schema(type="STRING", description="What to say to the business user"),
        "proposals": types.Schema(type="ARRAY", max_items=4, items=APPROACH_SCHEMA,
                                  description="Current proposals; empty once one is locked in"),
        "chosen": APPROACH_SCHEMA,  # nullable via not being required
    },
    required=["reply"])

NEGOTIATION_SYSTEM = """You are a customer-experience consultant helping a business owner design a 45-second outbound feedback call. You are talking TO the owner, not to their customer.

Business: {business_name}
Description: {business_description}
Customers being called: {target_user}
Desired call tone: {tone}
What they want to learn: {feedback_objective}

THE CHANNEL -- this constrains every idea you have
This is a live phone call and nothing else. The customer is holding a phone to their ear. They have exactly two ways to answer you:
  1. SPEAKING -- captured by speech-to-text. Good for open-ended answers and for numbers said aloud.
  2. PRESSING KEYS on the phone keypad (DTMF) -- digits 0-9, star, hash. Good for ratings and yes/no, and it survives noisy lines and heavy accents where speech recognition fails.
There is NO screen. Never propose a thumbs up or thumbs down, a button, a star to tap, a link, an emoji, a text message, an email, or a survey form. If an idea needs anything visual, it is not possible -- convert it to speech or a keypress.

Keypad rules when you use one:
- Only single digits are one keypress. A 0-10 scale cannot use "10" -- either say "press 0 for ten", or use a 1-9 range, or take that answer by speech instead.
- Always state the mapping out loud inside sample_question, e.g. "press 1 for yes, or 2 for no".
- Keep the mapping to at most 5 options; nobody remembers more while holding a phone.
- Open-ended answers must be input_mode "speech" -- a keypad cannot capture an opinion.

Your first message: propose 2-4 genuinely different approaches. Vary the input_mode across them -- at least one keypad-based and at least one spoken -- so the owner can weigh a crisp measurable answer against a richer one. Each must be answerable in under 15 seconds, and each must clearly serve what they want to learn. Keep "reply" to one short line -- the proposals are rendered separately, so do not restate them in prose.

After that, read what the owner says and act on it:
- Clear approval ("ok", "yes", "sounds good", "the second one"): set "chosen" to that exact approach and leave "proposals" empty.
- A tweak ("make it about ambience", "shorter", "use a 1 to 10 scale"): apply it. If the result is now unambiguous, put it straight in "chosen" rather than making them approve again.
- A different direction entirely: drop your earlier ideas and propose fresh ones that fit what they actually asked for.
- Something ambiguous or outside what a 45-second call can do: say so plainly in one line and offer the closest workable thing.

Rules:
- Never present the choices as a numbered list inside "reply"; they are shown from "proposals".
- sample_question must be phrased for the ear: one sentence, under 20 spoken words, no lists, digits as digits, and it must include the keypress instruction whenever input_mode is "keypad".
- If the owner asks for something the channel cannot do (a thumbs up, a star rating they can tap, a link), say plainly in one line that a phone call has no screen, then offer the closest keypad or spoken equivalent.
- Set "chosen" the moment the owner has settled on something. Do not keep asking for confirmation.
- Once "chosen" is set, "reply" is a single sentence confirming what the call will ask and how the customer answers it.
"""


MODE_LABEL = {"speech": "spoken answer", "keypad": "keypad press", "either": "spoken or keypad"}

def show_turn(turn: dict) -> None:
    print(turn["reply"])
    for p in turn.get("proposals") or []:
        print(f"\n  - {p['name']} ({p['dimension_type']}, {MODE_LABEL.get(p['input_mode'], p['input_mode'])})")
        print(f"    {p['description']}")
        print(f"    asks: \"{p['sample_question']}\"")
        if p.get("keypad_map"):
            print(f"    keys: {p['keypad_map']}")


def negotiate_approach(profile: dict, max_rounds: int = 8) -> dict:
    """Talk to the model in plain English until a single approach is agreed on."""
    chat = client.chats.create(
        model=PLANNING_MODEL,
        config=json_config(NEGOTIATION_SCHEMA, 2048,
                           system=NEGOTIATION_SYSTEM.format(**profile), temperature=0.6))

    # In TEST_MODE, exercise the refine path (not just instant approval) so the
    # negotiation is actually demonstrated end to end. The middle turn asks for
    # something the channel cannot do, to check the model pushes back instead of
    # cheerfully agreeing to a thumbs-up button on a phone call.
    scripted = ["Can they just give a thumbs up or thumbs down?",
                "Alright. What I really care about is whether they'd come back -- ask that.",
                "Good, use the keypad one."]

    def send(msg: str) -> dict:
        return json.loads(with_retry(lambda: chat.send_message(msg)).text)

    turn = send("Propose some approaches.")
    for _ in range(max_rounds):
        show_turn(turn)
        if turn.get("chosen"):
            return turn["chosen"]
        if TEST_MODE:
            if not scripted:
                raise RuntimeError("TEST_MODE script ran out before an approach was agreed.")
            reply = scripted.pop(0)
            print(f"\n> {reply}")
        else:
            reply = input("\n> ").strip() or "Yes, that works. Use it."
        turn = send(reply)

    raise RuntimeError("Could not settle on an approach; restart this cell and be more specific.")


selected_strategy = negotiate_approach(business_profile)
print("\n" + "=" * 60)
print("Locked in:", selected_strategy["name"], f"({selected_strategy['dimension_type']})")
print("Answered :", MODE_LABEL.get(selected_strategy["input_mode"], selected_strategy["input_mode"]))
print("Will ask :", selected_strategy["sample_question"])
if selected_strategy.get("keypad_map"):
    print("Keys     :", selected_strategy["keypad_map"])
print("Scoring  :", selected_strategy["score_guidance"])

## 3. Call LanguageType it however you like -- "Hindi", "brazilian portuguese", "Spanish but casual". The model resolves it to a language name and BCP-47 code for the STT/TTS engines, and writes the fallback goodbye line in that language now, so a hard cutoff mid-call never has to wait on a model round trip.

In [ ]:
LANGUAGE_SCHEMA = types.Schema(
    type="OBJECT",
    properties={
        "language_name": types.Schema(type="STRING", description="English name of the language, e.g. 'Hindi'"),
        "language_code": types.Schema(type="STRING", description="BCP-47 code for STT/TTS, e.g. 'hi' or 'pt-BR'"),
        "closing_line": types.Schema(type="STRING", description="A warm 10-word goodbye IN that language, in its native script"),
    },
    required=["language_name", "language_code", "closing_line"])

LANGUAGE_PROMPT = """The business owner wants the feedback call conducted in: "{raw}"

Resolve that to a language for a speech engine. Honour regional intent (e.g. "brazilian portuguese" -> pt-BR).
closing_line must be a natural spoken sign-off thanking the customer for their time, written in that language's
native script, under 10 words. If the request is unclear or not a language, fall back to English."""

language_request = ask("What language should the call be in?", "English")
CALL_LANGUAGE = call_json(PLANNING_MODEL, LANGUAGE_PROMPT.format(raw=language_request),
                          LANGUAGE_SCHEMA, max_tokens=512)

LANG_CODE = CALL_LANGUAGE["language_code"]
LANG_NAME = CALL_LANGUAGE["language_name"]
FALLBACK_CLOSING = CALL_LANGUAGE["closing_line"]

print(f"Call language: {LANG_NAME} ({LANG_CODE})")
print(f"Fallback closing: {FALLBACK_CLOSING}")

## 3b. Feedback DepthHow many feedback questions the agent may ask inside the 45s. Two is the practical ceiling for a call this short.

In [ ]:
MAX_QUESTIONS = ask_int("Max feedback questions to ask in the call (1 or 2)", 3, 1, 3)
print(f"Max feedback questions per call: {MAX_QUESTIONS}")

## 4. STT / TTS Engines (pluggable)Mock engines run over the console so the whole flow is runnable today. Once `STT_API_KEY` / `TTS_API_KEY` are set, implement `transcribe()` / `synthesize()` in the real subclasses -- the dialogue manager below never changes.Details that matter for a real swap-in:- **`listen()` returns speech *and* DTMF digits**, because a phone leg can deliver either at any moment. Wire `text` to your STT stream and `digits` to your telephony provider's DTMF/gather events (Twilio, Vonage, Plivo all emit these). Always keep speech capture armed even on a keypad question -- callers ignore instructions.- `synthesize()` is called with **partial** text as the model streams, so a real TTS client should stream audio out rather than buffer the whole reply.- The mock reports how long it actually waited; the dialogue manager subtracts human typing time from the call budget so the 45s timer stays meaningful in a console demo (a real engine returns true wall-clock latency and nothing is subtracted).In the console mock, **type digits to simulate a keypress** (`4`) and anything else to simulate speech.

In [ ]:
@dataclass
class CustomerInput:
    """What came back from the caller on one turn. A phone gives us exactly two
    channels, and a real telephony leg can deliver either at any moment -- a caller
    may press a key while the prompt is still playing, or just answer out loud."""
    text: str = ""     # transcribed speech
    digits: str = ""   # DTMF keypresses, e.g. "4"

    @property
    def empty(self) -> bool:
        return not (self.text or self.digits)

    def as_model_message(self) -> str:
        """How this turn is described to the model."""
        if self.digits and self.text:
            return f'<keypad: {self.digits}> {self.text}'
        if self.digits:
            return f'<keypad: {self.digits}>'
        return self.text or "<no response>"

    def as_transcript(self) -> str:
        if self.empty:
            return "<silence>"
        if self.digits and self.text:
            return f'[pressed {self.digits}] {self.text}'
        if self.digits:
            return f'[pressed {self.digits}]'
        return self.text


class STTEngine:
    def listen(self, language: str, timeout_s: float, expect_digits: bool) -> CustomerInput:
        """Listen for up to timeout_s seconds. Return speech, DTMF digits, or neither.
        `expect_digits` is a hint that the current question asked for a keypress -- a real
        implementation should still accept speech, since callers ignore instructions."""
        raise NotImplementedError

    def billable_seconds(self) -> float:
        """Wall-clock seconds of the last listen() that should count against the
        call budget. Real engines: all of it. Console mock: none (human typing time)."""
        return 0.0


class TTSEngine:
    def synthesize(self, text: str, language: str, final: bool = True) -> None:
        """Speak `text`. Called repeatedly with growing text as the model streams;
        `final=True` marks the last chunk of this turn."""
        raise NotImplementedError


class ConsoleMockSTT(STTEngine):
    """Stand-in for a real telephony leg. Replace with e.g. Twilio/Deepgram once STT_API_KEY
    is set: speech from the STT stream, digits from the DTMF/gather events.

    In the console, a line that is only digits is treated as keypresses, so you can type
    `4` to simulate pressing 4. Typing time is excluded from the call budget so the 45s
    timer still means something."""
    def __init__(self, scripted: Optional[List[str]] = None):
        self.scripted = list(scripted or [])
        self._elapsed = 0.0

    def listen(self, language: str, timeout_s: float, expect_digits: bool) -> CustomerInput:
        t0 = time.perf_counter()
        if self.scripted:
            raw = self.scripted.pop(0)
        else:
            hint = "press keys or speak" if expect_digits else "speak"
            raw = input(f"[customer -- {hint}] ")
        raw = raw.strip()
        self._elapsed = time.perf_counter() - t0

        if raw and all(c in "0123456789*#" for c in raw):
            result = CustomerInput(digits=raw)
        else:
            result = CustomerInput(text=raw)
        if self.scripted or raw:
            print(f"[customer] {result.as_transcript()}")
        return result

    def billable_seconds(self) -> float:
        return 0.0


class ConsoleMockTTS(TTSEngine):
    """Stand-in for a real TTS API. Replace with e.g. ElevenLabs/Google TTS once TTS_API_KEY is set.
    Prints incrementally so streamed turns are visible as they arrive."""
    def __init__(self):
        self._spoken = 0

    def synthesize(self, text: str, language: str, final: bool = True) -> None:
        if self._spoken == 0 and text:
            print(f"[AI ({language})]: ", end="", flush=True)
        if len(text) > self._spoken:
            print(text[self._spoken:], end="", flush=True)
            self._spoken = len(text)
        if final:
            print(flush=True)
            self._spoken = 0


# Scripted customer for TEST_MODE: covers an identity question, an off-topic detour,
# a keypress answer, a spoken elaboration, and silence -- every branch the dialogue
# manager handles. A bare-digits line simulates a DTMF keypress.
SCRIPTED_CUSTOMER = [
    "Wait, who is this exactly?",
    "Okay. By the way, what time do you close today?",
    "4",
    "The food was great but I waited a while.",
    "",
]


def build_stt_engine() -> STTEngine:
    if STT_API_KEY:
        raise NotImplementedError("Plug in real STT client here using STT_API_KEY")
    return ConsoleMockSTT(SCRIPTED_CUSTOMER if TEST_MODE else None)

def build_tts_engine() -> TTSEngine:
    if TTS_API_KEY:
        raise NotImplementedError("Plug in real TTS client here using TTS_API_KEY")
    return ConsoleMockTTS()

stt_engine = build_stt_engine()
tts_engine = build_tts_engine()
print("Engines ready:", type(stt_engine).__name__, "/", type(tts_engine).__name__)

## 5. Dialogue ManagerRuns the actual call: builds a system prompt from the business profile + agreed approach + language, then drives a turn-by-turn Gemini conversation inside a **hard 45-second budget**. Handles silence, off-topic/irrelevant questions, and identity questions ("who are you") in real time via the system prompt, and force-ends the call when time runs out.The reply schema is enforced by the API (`response_schema`), so the prompt spends its words on *behavior* instead of on describing a JSON shape the model might still get wrong.

In [ ]:
CALL_DURATION_S = 45
TURN_LISTEN_TIMEOUT_S = 8
MAX_SILENCE_STRIKES = 2      # 2 silences -> re-prompt, 3rd -> hang up
MAX_TURNS = 8                # hard safety cap regardless of the timer
TURN_MAX_TOKENS = 160        # a spoken turn is ~25 words; keeps generation fast

END_REASONS = ["completed", "silence", "off_topic", "hostile", "declined"]

TURN_SCHEMA = types.Schema(
    type="OBJECT",
    properties={
        "speech": types.Schema(type="STRING"),
        "should_end": types.Schema(type="BOOLEAN"),
        "end_reason": types.Schema(type="STRING", nullable=True, enum=END_REASONS),
        "score": types.Schema(type="NUMBER", nullable=True),
        "sentiment": types.Schema(type="STRING", nullable=True, enum=["positive", "neutral", "negative"]),
    },

    # `speech` first is deliberate: the schema order is the generation order, so the
    # spoken text streams out before the metadata fields and TTS can start early.
    required=["speech", "should_end"])

SYSTEM_PROMPT_TEMPLATE = """You are a voice agent phoning a customer of "{business_name}" for feedback.
Tone: {tone}. Speak only {language_name}.

Approach: {strategy_name} ({dimension_type}) -- {strategy_description}
Question to work from: "{sample_question}"
How the customer answers: {input_mode_line}
What the business wants to learn: {feedback_objective}

THE CHANNEL
You are a voice on a phone. The customer has no screen, so never refer to anything they could look at, tap, or click, and never mention a thumbs up, a button, a link, or a text message. They can only speak or press keys.
A message like "<keypad: 4>" means they pressed 4 on their phone. Treat that as their answer -- acknowledge it naturally in words ("Got it, a 4") and never ask them to repeat it out loud.
If they speak a number when you asked for a keypress, or press a key when you asked them to speak, accept it either way. Never correct them about how they answered.
If they press a key that is not in the mapping you gave, restate the valid keys once, briefly.

CALL SHAPE
- Turn 1: one short greeting clause naming the business, then your first feedback question. No small talk.
- Ask AT MOST {max_questions} feedback question(s) all call. A silence re-prompt or a one-line redirect is not a new question.
- The instant you have answers to {max_questions} question(s), thank them and end. Never keep the line open to be polite.
- The whole call must fit in 45 seconds, so every reply is at most 2 sentences and under 25 spoken words.

VOICE STYLE
- Write words a person says out loud: no markdown, no bullet points, no emoji, no stage directions.
- Say digits as digits ("rate us 1 to 5"). Never read the approach name or any internal label aloud.
- When the question needs a keypress, say the mapping plainly and briefly ("press 1 for yes, 2 for no"), and say it only once unless they press something invalid.
- Do not repeat a question the customer already answered, and never re-introduce yourself twice.

HANDLING THE CUSTOMER
- "<no response>" means silence, unintelligible audio, or no keypress. Re-prompt once, shorter and simpler -- if the question had a keypad option, lead with that on the retry, since it works when the line is noisy. On a second silence, say a brief goodbye and set should_end with end_reason "silence".
- Asked who you are or how this works: answer in one short clause ("an automated feedback call for {business_name}"), then immediately re-ask your question in the same turn.
- Off-topic (hours, prices, an order problem, wants a human, small talk): you do NOT know any business fact beyond what is written above, so never answer the question itself. Say you do not have that detail but a team member can follow up, then re-ask in the same turn. If they derail a second time, end with end_reason "off_topic".
- Hostile, abusive, or a clear refusal: apologize once, thank them, end immediately with end_reason "hostile" or "declined".
- A vague answer ("it was fine") to a scored question: ask once for the number, and only once.
- You have no access to hours, prices, menu, staff, or this customer's order. Saying "I don't have that in front of me" is always correct; guessing never is. Never promise a discount, refund, or callback time.
- Anything unexpected: use your best judgment in the spirit of these rules.

FIELDS
- should_end true only when the speech you are returning IS the goodbye. Set end_reason only then.
- score: {score_guidance} Report it ONLY once the customer has actually given that answer -- a valid keypress, or the number/answer spoken aloud. An invalid keypress, a vague remark, or a question from them is not an answer: leave score null. Never guess a score to fill the field. Once they have genuinely answered, repeat that same score on every later turn.
- sentiment reflects the customer's tone so far, or null before they have said anything substantive.
"""


@dataclass
class CallResult:
    transcript: List[Dict[str, str]] = field(default_factory=list)
    end_reason: str = ""
    score: Optional[float] = None
    sentiment: Optional[str] = None
    keypresses: List[str] = field(default_factory=list)
    turn_latencies_s: List[float] = field(default_factory=list)
    duration_s: float = 0.0


INPUT_MODE_LINES = {
    "speech": "By speaking. Do not ask for a keypress.",
    "keypad": "By pressing a key on the phone keypad. {keys} State the mapping when you ask.",
    "either": "Either by speaking or by pressing a key. {keys} Offer the keypress option, but accept a spoken answer just as readily.",
}


def build_system_prompt(profile: dict, strategy: dict, language_name: str, max_questions: int) -> str:
    mode = strategy.get("input_mode", "speech")
    keys = f"Key mapping: {strategy['keypad_map']}." if strategy.get("keypad_map") else ""
    return SYSTEM_PROMPT_TEMPLATE.format(
        business_name=profile["business_name"],
        tone=profile["tone"],
        feedback_objective=profile["feedback_objective"],
        language_name=language_name,
        strategy_name=strategy["name"],
        dimension_type=strategy["dimension_type"],
        strategy_description=strategy["description"],
        sample_question=strategy["sample_question"],
        input_mode_line=INPUT_MODE_LINES.get(mode, INPUT_MODE_LINES["speech"]).format(keys=keys).strip(),
        max_questions=max_questions,
        score_guidance=strategy["score_guidance"],
    )


_SPEECH_RE = re.compile(r'"speech"\s*:\s*"((?:[^"\\]|\\.)*)')

def _partial_speech(buf: str) -> str:
    """Best-effort extract of the (possibly still-growing) `speech` string from partial JSON."""
    m = _SPEECH_RE.search(buf)
    if not m:
        return ""
    try:
        return json.loads(f'"{m.group(1)}"')
    except json.JSONDecodeError:
        return ""

In [ ]:
def run_call(profile: dict, strategy: dict, language_name: str, language_code: str,
             fallback_closing: str, max_questions: int,
             stt: STTEngine, tts: TTSEngine) -> CallResult:
    system_prompt = build_system_prompt(profile, strategy, language_name, max_questions)
    chat = client.chats.create(
        model=TURN_MODEL,
        config=json_config(TURN_SCHEMA, TURN_MAX_TOKENS, system=system_prompt, temperature=0.4))

    result = CallResult()
    started = time.perf_counter()
    budget_used = 0.0          # seconds of real airtime consumed
    silence_strikes = 0
    turns = 0
    # Hint for the input engine: a real telephony leg needs to know whether to arm
    # DTMF capture for this question. It still accepts speech either way.
    expects_digits = strategy.get("input_mode") in ("keypad", "either")

    def remaining() -> float:
        return CALL_DURATION_S - (time.perf_counter() - started - budget_used)

    def send_turn(message: str) -> dict:
        """Stream one turn: speak `speech` as soon as it arrives, then return the parsed object.
        Falls back to a plain hang-up if the model output is unusable, instead of raising."""
        t0 = time.perf_counter()
        buf, spoken = "", ""
        try:
            for chunk in chat.send_message_stream(message):
                buf += chunk.text or ""
                partial = _partial_speech(buf)
                if len(partial) > len(spoken):
                    tts.synthesize(partial, language_code, final=False)
                    spoken = partial
            turn_json = json.loads(buf)
        except (json.JSONDecodeError, ValueError, TypeError):
            turn_json = {"speech": fallback_closing, "should_end": True, "end_reason": "error"}
        latency = time.perf_counter() - t0
        result.turn_latencies_s.append(latency)
        # Flush anything the stream held back (or the fallback text) and close the line.
        tts.synthesize(turn_json.get("speech", ""), language_code, final=True)
        if VERBOSE_LATENCY:
            print(f"    (turn latency {latency:.2f}s)")
        return turn_json

    def record(turn_json: dict) -> None:
        result.transcript.append({"role": "ai", "text": turn_json.get("speech", "")})
        if turn_json.get("score") is not None:
            result.score = turn_json["score"]
        if turn_json.get("sentiment"):
            result.sentiment = turn_json["sentiment"]

    def hard_close(reason: str) -> None:
        # No-LLM-call fallback, so a hard cutoff never waits on another round trip.
        tts.synthesize(fallback_closing, language_code, final=True)
        result.transcript.append({"role": "ai", "text": fallback_closing})
        result.end_reason = reason

    def finish(reason: str) -> CallResult:
        result.end_reason = result.end_reason or reason
        result.duration_s = time.perf_counter() - started - budget_used
        lat = result.turn_latencies_s
        avg = f"{sum(lat) / len(lat):.2f}s" if lat else "n/a"
        print(f"\n--- Call ended: {result.end_reason} | score={result.score} | "
              f"sentiment={result.sentiment} | airtime={result.duration_s:.1f}s | "
              f"turns={len(lat)} | avg turn {avg} ---")
        return result

    opening = send_turn("Begin the call now with your greeting and first question.")
    record(opening)
    if opening.get("should_end"):
        return finish(opening.get("end_reason") or "error")

    while True:
        if remaining() <= 1:
            hard_close("time_up")
            break
        if turns >= MAX_TURNS:
            hard_close("max_turns")
            break

        t_listen = time.perf_counter()
        customer = stt.listen(language_code, min(TURN_LISTEN_TIMEOUT_S, remaining()),
                              expect_digits=expects_digits)
        listened = time.perf_counter() - t_listen
        # Only the engine's real listening time counts against the 45s; the console
        # mock reports 0 so human typing does not blow the budget in a demo.
        budget_used += max(0.0, listened - min(stt.billable_seconds(), listened))
        turns += 1

        result.transcript.append({"role": "customer", "text": customer.as_transcript()})
        if customer.digits:
            result.keypresses.append(customer.digits)

        if customer.empty:
            silence_strikes += 1
            if silence_strikes > MAX_SILENCE_STRIKES:
                hard_close("silence")
                break
            reply = send_turn("<no response>")
            record(reply)
            if reply.get("should_end"):
                result.end_reason = reply.get("end_reason") or "silence"
                break
            continue

        silence_strikes = 0
        reply = send_turn(customer.as_model_message())
        record(reply)
        if reply.get("should_end"):
            result.end_reason = reply.get("end_reason") or "completed"
            break

    return finish("completed")

## 6. Run the Call

In [ ]:
call_result = run_call(business_profile, selected_strategy, LANG_NAME, LANG_CODE,
                       FALLBACK_CLOSING, MAX_QUESTIONS, stt_engine, tts_engine)
call_result.transcript

## 7. Extract Structured FeedbackTurn the raw transcript into a structured result. The scoring rule comes from the approach the model itself designed in Section 2, so this step adapts automatically to whatever scale was agreed on. The score the agent tracked live is passed in as a cross-check -- the summarizer is told to trust the transcript over it.

In [ ]:
SUMMARY_SCHEMA = types.Schema(
    type="OBJECT",
    properties={
        "score": types.Schema(type="NUMBER", nullable=True),
        "sentiment": types.Schema(type="STRING", enum=["positive", "neutral", "negative"]),
        "summary": types.Schema(type="STRING"),
        "key_points": types.Schema(type="ARRAY", items=types.Schema(type="STRING"), max_items=5),
        "answered": types.Schema(type="BOOLEAN"),
        "follow_up_needed": types.Schema(type="BOOLEAN"),
    },
    required=["score", "sentiment", "summary", "key_points", "answered", "follow_up_needed"])

SUMMARY_PROMPT = """Extract a structured result from this customer feedback call.

Approach used: {strategy_name} ({dimension_type}) -- {strategy_description}
How the customer answered: {input_mode}. {keypad_map}
Scoring rule: {score_guidance}
Call ended because: {end_reason}
Score the agent tracked live (may be wrong -- the transcript wins): {live_score}
Keys the customer pressed, in order: {keypresses}

Transcript:
{transcript}

Rules:
- This was a phone call. "[pressed 4]" in the transcript means the customer pressed 4 on their keypad -- that IS their answer, as valid as speaking it, so read it through the key mapping above.
- Use ONLY what the customer actually said or pressed. Never infer a number they did not give: if they gave no usable rating, score is null.
- summary: 1-2 sentences, factual, no praise or padding.
- key_points: at most 5 short phrases, each a distinct piece of feedback about the business. Do not restate the score, and exclude off-topic remarks and questions the customer asked. Empty list if they gave no substantive feedback.
- answered: true only if the customer gave a real answer to at least one feedback question.
- follow_up_needed: true if the customer voiced ANY dissatisfaction, complaint, or unresolved issue (a long wait, a mistake, cold food), or asked for a human -- even alongside praise and even with a high score.
"""

def summarize_feedback(result: CallResult, strategy: dict) -> dict:
    transcript_text = "\n".join(f"{t['role']}: {t['text']}" for t in result.transcript)
    prompt = SUMMARY_PROMPT.format(
        strategy_name=strategy["name"],
        dimension_type=strategy["dimension_type"],
        strategy_description=strategy["description"],
        input_mode=MODE_LABEL.get(strategy.get("input_mode", "speech"), "spoken answer"),
        keypad_map=f"Key mapping: {strategy['keypad_map']}." if strategy.get("keypad_map") else "",
        score_guidance=strategy["score_guidance"],
        end_reason=result.end_reason,
        live_score=result.score,
        keypresses=", ".join(result.keypresses) or "none",
        transcript=transcript_text,
    )
    data = call_json(PLANNING_MODEL, prompt, SUMMARY_SCHEMA, max_tokens=1024)
    data["call_ended_reason"] = result.end_reason
    data["dimension_type"] = strategy["dimension_type"]
    return data

feedback_summary = summarize_feedback(call_result, selected_strategy)
feedback_summary